# Instrumental Variables

This notebook uses a function for 2SLS and illustrates it by redoing an empirical example.

## Load Packages and Extra Functions

The key functions used for OLS and IV/2SLS are from the (local) `FinEcmt_OLS` module.

In [1]:
MyModulePath = joinpath(pwd(),"src")
!in(MyModulePath,LOAD_PATH) && push!(LOAD_PATH,MyModulePath)
using FinEcmt_OLS

In [2]:
#=
include(joinpath(pwd(),"src","FinEcmt_OLS.jl"))
using .FinEcmt_OLS
=#

In [3]:
using DelimitedFiles, LinearAlgebra

# Loading the Data

The next cells uses data from Mroz (1987) to replicate old results. See also Greene (2018) and Hill et al (2018). The chapter gives more details.

### A remark on the code
The data set contains many different variables. To import them to a named tuple, we use the `Readcsv()` from the `FinEcmt_OLS` module. (This is convenient, but not important for the focus of this notebook. An alternative is to use the `DataFrames.jl` package.)

In [4]:
X = Readcsv("Data/mroz.csv",0;ToFloat=true);

println("Variables in X: ",keys(X))
T = length(X.wage)
c = ones(T);              #create a constant

Variables in X: (:taxableinc, :federaltax, :hsiblings, :hfathereduc, :hmothereduc, :siblings, :lfp, :hours, :kidsl6, :kids618, :age, :educ, :wage, :wage76, :hhours, :hage, :heduc, :hwage, :faminc, :mtr, :mothereduc, :fathereduc, :unemployment, :bigcity, :exper)


## OLS

estimation of the log wage on education, experience and experience^2. Only data points where wage > 0 are used.

In [5]:
vv = X.wage .> 0         #find data points where X.wage > 0
                         #OLS of log wage, for observations with wage>0
(b_OLS,_,_,Covb,) = OlsNW(log.(X.wage[vv]),[c X.exper X.exper.^2 X.educ][vv,:])
Stdb_ols = sqrt.(diag(Covb))

colNames = ["coef","t-stat"]
rowNames = ["c","exper","exper^2","educ",]
printblue("OLS estimates:\n")
printmat(b_OLS,b_OLS./Stdb_ols;colNames,rowNames,prec=4)

OLS estimates:

             coef    t-stat
c         -0.5220   -2.6010
exper      0.0416    2.7344
exper^2   -0.0008   -1.9402
educ       0.1075    8.1697



## IV (2SLS)

using the function `TwoSLS()` function.

In this application, the mother's education is used as an instrument for the person's education.

In [6]:
@doc2 TwoSLS

```julia
TwoSLS(y,x,z,NWQ=true,m=0)
```

### Input

  * `y::VecOrMat`:      Tx1 or T-vector of the dependent variable
  * `x::VecOrMat`:      Txk matrix (or vector) of regressors
  * `z::VecOrMat`:      TxL matrix (or vector) of instruments
  * `NWQ:Bool`:         if true, then Newey-West's covariance matrix is used, otherwise Gauss-Markov
  * `m::Int`:           scalar, bandwidth in Newey-West; 0 means White's method

### Output

  * `b::Vector`:             k-vector, regression coefficients
  * `fnOutput::NamedTuple`:  with

      * res                Tx1 or Txn matrix, residuals y - yhat
      * yhat               Tx1 or Txn matrix, fitted values
      * Covb               matrix, covariance matrix of vec(b) = [beq1;beq2;...]
      * R2                 1xn, R2
      * R2_stage1          k-vector, R2 of each x[:,i] in first stage regression on z
      * δ_stage1           Lxk matrix, coeffs from 1st stage x = z'δ
      * Stdδ_stage1        Lxk matrix, std of δ

### Requires

  * Statistics, LinearAlgebra
  * CovNW


In [7]:
#using CodeTracking
#println(@code_string TwoSLS([1],[1],[1]))    #print the source code

In [8]:
(b_iv,fO2) = TwoSLS(log.(X.wage[vv]),[c X.exper X.exper.^2 X.educ][vv,:],
                                     [c X.exper X.exper.^2 X.mothereduc][vv,:])

zNames = ["c","exper","exper^2","mothereduc"]

printblue("first-stage estimates: coeffs (each regression in its own column)")
printmat(fO2.δ_stage1;colNames=rowNames,rowNames=zNames)

tstats = fO2.δ_stage1./fO2.Stdδ_stage1
printblue("first-stage estimates: t-stats")
printmat(tstats[:,4];colNames=rowNames[2:2],rowNames=zNames)     #2:2 to make it a vector

printblue("first-stage estimates: R²")
printmat(fO2.R2_stage1';colNames=rowNames)

first-stage estimates: coeffs (each regression in its own column)
                   c     exper   exper^2      educ
c              1.000    -0.000     0.000     9.775
exper          0.000     1.000    -0.000     0.049
exper^2       -0.000     0.000     1.000    -0.001
mothereduc     0.000    -0.000    -0.000     0.268

first-stage estimates: t-stats
               exper
c             23.753
exper          1.152
exper^2       -0.959
mothereduc     8.481

first-stage estimates: R²
         c     exper   exper^2      educ
       NaN     1.000     1.000     0.153



In [9]:
t_iv = b_iv./sqrt.(diag(fO2.Covb))
printblue("IV estimates")
printmat(b_iv,t_iv;colNames=["coef","t-stat"],rowNames)

printblue("DWH test (Chisq(1)): ")
printlnPs(fO2.DWHtest)

IV estimates
             coef    t-stat
c           0.198     0.407
exper       0.045     2.888
exper^2    -0.001    -2.145
educ        0.049     1.301

DWH test (Chisq(1)): 
     2.851
